In [1]:
%load_ext autoreload

In [ ]:
%autoreload 2
import os
import logging
from pathlib import Path
from shapely import wkt
from geoalchemy2 import Geometry
from sqlalchemy import TIMESTAMP, BigInteger, Integer

from darpinstances.instance_generation.demand_to_database import load_data_from_csv, load_demand_to_db
from darpinstances.instance_generation.demand_generation import generate_demand
from darpinstances.instance_generation.vehicles import generate_vehicles
from darpinstances.instance_generation.convert_formats import RESOURCE_PATH, generate_and_save_csv
from darpinstances.instance_generation.map import NearestNodeProvider, get_map
from darpinstances.instance_generation.instance_generation import generate_dm
from darpinstances.instance import load_instance_config
from roadgraphtool import db
from roadgraphtool.config import parse_config_file

CONFIG_DB = Path('/home/dominika/Desktop/smart-mobility/road-graph-tool/config.yml')
PATH = Path.cwd()
INSTANCE_PATH = PATH.parents[2] / "Instances"
RESULTS_PATH = PATH.parents[2] / "Results"

Set up database connection

In [12]:
config_db = parse_config_file(CONFIG_DB)
db.init_db(config_db)

In [4]:
db.db.ssh_tunnel_local_port

1114

## Generate demands from source files and save to CSV

### Porto, Portugal

In [ ]:
generate_and_save_csv("Porto")

### Sydney, Australia

In [ ]:
generate_and_save_csv("Sydney")

## Import CSV demands to database (optional)

### Porto, Portugal

In [2]:
city = "Porto"

In [15]:
csvfile = RESOURCE_PATH / f'{city}_trips.csv'
df = load_data_from_csv(csvfile)
# load_demand_to_db(df, "trips", city.lower(), {'origin': Geometry('POINT', srid=4326), 'destination': Geometry('POINT', srid=4326), 'timestamp': TIMESTAMP})

### Sydney, Australia

In [1]:
city = "Sydney"

In [5]:
csvfile = RESOURCE_PATH / f'{city}_trips.csv'
df = load_data_from_csv(csvfile)

In [27]:
load_demand_to_db(df, "trips", city.lower(), {'origin': Geometry('POINT', srid=4326), 'destination': Geometry('POINT', srid=4326), 'timestamp': TIMESTAMP})

11:27:47 [INFO] Starting _psycopg2 connection
11:27:47 [INFO] Starting sql_alchemy connection


Loading data into the database...
Data loaded into the database successfully.


## Map demand positions to existing nodes

Generate instance configurations

In [5]:
from darpinstances.instance_generation.generate_config import generate_config, default_config

def create_custom_instance_config(area_id, max_prolongation, srid_plane, min_time, max_time, start_time, dataset, positions_set, time_set, capacity):
    default_config['area_id'] = area_id
    default_config['area_dir'] = "../../../../"
    default_config['max_prolongation'] = max_prolongation
    default_config['map']['SRID_plane'] = srid_plane
    default_config['map']['path'] = "../../../../map"
    default_config['demand']['filepath'] = "./trips.di"
    default_config['demand']['mode'] = 'load'
    default_config['demand']['min_time'] = min_time
    default_config['demand']['max_time'] = max_time
    default_config['demand']['dataset'] = dataset
    default_config['demand']['positions_set'] = positions_set
    default_config['demand']['time_set'] = time_set
    default_config['vehicles']['start_time'] = start_time
    default_config['vehicles']['vehicle_capacity'] = capacity
    return default_config

In [7]:
def create_custom_results_config(instance_path, method, outdir):
    config = {}
    config['instance'] = instance_path
    config['method'] = method
    config['outdir'] = outdir
    return config

In [22]:
from datetime import datetime, timedelta
cities = ['Porto', 'Sydney', 'NYC', 'DC', 'Chicago', 'Manhattan']
start_durations_str = {'18-00': ['05_min', '15_min', '30_min', '2_h']}
# start_durations_str = {'18-00': ['01_min', '02_min', '05_min', '15_min', '30_min', '2_h']}
# start_durations_str = {'07-00': ['16_h'], '18-00': ['2_h', '01_min', '30_min', '15_min', '05_min']}
start_durations = {18: [5, 15, 30, 120]}
# start_durations = {7: [16*60], 18: [180, 1, 30, 15, 5]}
# delays_str = ['03_min', '05_min', '10_min']
delays_str = ['03_min', '05_min', '10_min', '15_min']
# delays = [3, 5, 10]
delays = [3, 5, 10, 15]
methods = ['ih', 'vga', 'halns-vga', 'vga_chaining']
# methods = ['ih', 'vga', 'halns', 'halns-ih', 'halns-vga', 'vga_chaining']
capacities = [4, 6, 10]

In [23]:
from itertools import product
c = 0
for start, durations in start_durations.items():
    min_time = datetime(2014, 1, 1, 18, 0, 0)
    start_str = f"{start:02d}-00"
    for (i, duration), (j, delay), city, capacity in product(
        enumerate(durations), enumerate(delays), cities, capacities
    ):
                        
        max_time = min_time + timedelta(minutes=duration)
        start_time = min_time - timedelta(minutes=30)
        
        instance_dir = INSTANCE_PATH / f'{city}/instances/start_{start_str}/duration_{start_durations_str[start_str][i]}/max_delay_{delays_str[j]}/capacity_{capacity}'
        results_dir = RESULTS_PATH / f'{city}/start_{start_str}/duration_{start_durations_str[start_str][i]}/max_delay_{delays_str[j]}/capacity_{capacity}/{method}'
        
        os.makedirs(instance_dir, exist_ok=True)
        os.makedirs(results_dir, exist_ok=True)
        
        instance_path = instance_dir / "config.yaml"
        results_path = results_dir / "config.yaml"
        
        # set dataset, srid_plane...
        instance_config = create_custom_instance_config(
            1, 
            delay*60, 
            32756, 
            min_time.strftime('%Y-%m-%d %H:%M:%S'), 
            max_time.strftime('%Y-%m-%d %H:%M:%S'), 
            start_time.strftime('%Y-%m-%d %H:%M:%S'), 
            1, 
            1, 
            1, 
            capacity)
        for method in methods:
            results_config = create_custom_results_config(str(instance_path), method, str(results_dir))
        
        generate_config(instance_config, instance_path)
        generate_config(results_config, results_path)
        c += 1
print(f'{c} configs created')

12:01:45 [INFO] Saving config to /home/dominika/Desktop/deathOFbachelor/tests/Instances/Porto/instances/start_18-00/duration_05_min/max_delay_03_min/capacity_4/config.yaml
12:01:45 [INFO] Saving config to /home/dominika/Desktop/deathOFbachelor/tests/Results/Porto/start_18-00/duration_05_min/max_delay_03_min/capacity_4/vga_chaining/config.yaml
12:01:45 [INFO] Saving config to /home/dominika/Desktop/deathOFbachelor/tests/Instances/Porto/instances/start_18-00/duration_05_min/max_delay_03_min/capacity_6/config.yaml
12:01:45 [INFO] Saving config to /home/dominika/Desktop/deathOFbachelor/tests/Results/Porto/start_18-00/duration_05_min/max_delay_03_min/capacity_6/vga_chaining/config.yaml
12:01:45 [INFO] Saving config to /home/dominika/Desktop/deathOFbachelor/tests/Instances/Porto/instances/start_18-00/duration_05_min/max_delay_03_min/capacity_10/config.yaml
12:01:45 [INFO] Saving config to /home/dominika/Desktop/deathOFbachelor/tests/Results/Porto/start_18-00/duration_05_min/max_delay_03_min/

12:01:45 [INFO] Saving config to /home/dominika/Desktop/deathOFbachelor/tests/Results/DC/start_18-00/duration_05_min/max_delay_03_min/capacity_4/vga_chaining/config.yaml
12:01:45 [INFO] Saving config to /home/dominika/Desktop/deathOFbachelor/tests/Instances/DC/instances/start_18-00/duration_05_min/max_delay_03_min/capacity_6/config.yaml
12:01:45 [INFO] Saving config to /home/dominika/Desktop/deathOFbachelor/tests/Results/DC/start_18-00/duration_05_min/max_delay_03_min/capacity_6/vga_chaining/config.yaml
12:01:45 [INFO] Saving config to /home/dominika/Desktop/deathOFbachelor/tests/Instances/DC/instances/start_18-00/duration_05_min/max_delay_03_min/capacity_10/config.yaml
12:01:45 [INFO] Saving config to /home/dominika/Desktop/deathOFbachelor/tests/Results/DC/start_18-00/duration_05_min/max_delay_03_min/capacity_10/vga_chaining/config.yaml
12:01:45 [INFO] Saving config to /home/dominika/Desktop/deathOFbachelor/tests/Instances/Chicago/instances/start_18-00/duration_05_min/max_delay_03_min

288 configs created


In [19]:
count = 0
for start, durations in start_durations.items():
    for i, duration in enumerate(durations):
        for j, delay in enumerate(delays):
            for method in methods:
                for city in cities:
                    for capacity in capacities:
                        if start == 7:
                            min_time = datetime(2014, 1, 1, 7, 0, 0)
                            start_str = '07-00'
                        else:
                            min_time = datetime(2014, 1, 1, 18, 0, 0)
                            start_str = '18-00'
                        max_time = min_time + timedelta(minutes=duration)
                        start_time = min_time - timedelta(minutes=30)
                        
                        instance_dir = INSTANCE_PATH / f'{city}/instances/' / f'start_{start_str}' / f'duration_{start_durations_str[start_str][i]}' / f'max_delay_{delays_str[j]}'
                        results_dir = RESULTS_PATH / f'{city}' / f'start_{start_str}' / f'duration_{start_durations_str[start_str][i]}' / f'max_delay_{delays_str[j]}' / method
                        
                        os.makedirs(instance_dir, exist_ok=True)
                        os.makedirs(results_dir, exist_ok=True)
                        
                        instance_path = instance_dir / "config.yaml"
                        results_path = results_dir / "config.yaml"
                        
                        # set dataset, srid_plane...
                        instance_config = create_custom_instance_config(1, delay*60, 32756, min_time.strftime('%Y-%m-%d %H:%M:%S'), max_time.strftime('%Y-%m-%d %H:%M:%S'), start_time.strftime('%Y-%m-%d %H:%M:%S'), 1, 1, 1, capacity)
                        results_config = create_custom_results_config(str(instance_path), method, str(results_dir))
                        
                        generate_config(instance_config, instance_path)
                        generate_config(results_config, results_path)
                        count += 1
print(f'{count} configs created')

11:49:43 [INFO] Saving config to /home/dominika/Desktop/deathOFbachelor/tests/Instances/Porto/instances/start_18-00/duration_01_min/max_delay_03_min/config.yaml
11:49:43 [INFO] Saving config to /home/dominika/Desktop/deathOFbachelor/tests/Results/Porto/start_18-00/duration_01_min/max_delay_03_min/ih/config.yaml
11:49:43 [INFO] Saving config to /home/dominika/Desktop/deathOFbachelor/tests/Instances/Porto/instances/start_18-00/duration_01_min/max_delay_03_min/config.yaml
11:49:43 [INFO] Saving config to /home/dominika/Desktop/deathOFbachelor/tests/Results/Porto/start_18-00/duration_01_min/max_delay_03_min/ih/config.yaml
11:49:43 [INFO] Saving config to /home/dominika/Desktop/deathOFbachelor/tests/Instances/Porto/instances/start_18-00/duration_01_min/max_delay_03_min/config.yaml
11:49:43 [INFO] Saving config to /home/dominika/Desktop/deathOFbachelor/tests/Results/Porto/start_18-00/duration_01_min/max_delay_03_min/ih/config.yaml
11:49:43 [INFO] Saving config to /home/dominika/Desktop/death

11:49:43 [INFO] Saving config to /home/dominika/Desktop/deathOFbachelor/tests/Results/Sydney/start_18-00/duration_01_min/max_delay_03_min/ih/config.yaml
11:49:43 [INFO] Saving config to /home/dominika/Desktop/deathOFbachelor/tests/Instances/Sydney/instances/start_18-00/duration_01_min/max_delay_03_min/config.yaml
11:49:43 [INFO] Saving config to /home/dominika/Desktop/deathOFbachelor/tests/Results/Sydney/start_18-00/duration_01_min/max_delay_03_min/ih/config.yaml
11:49:43 [INFO] Saving config to /home/dominika/Desktop/deathOFbachelor/tests/Instances/NYC/instances/start_18-00/duration_01_min/max_delay_03_min/config.yaml
11:49:43 [INFO] Saving config to /home/dominika/Desktop/deathOFbachelor/tests/Results/NYC/start_18-00/duration_01_min/max_delay_03_min/ih/config.yaml
11:49:43 [INFO] Saving config to /home/dominika/Desktop/deathOFbachelor/tests/Instances/NYC/instances/start_18-00/duration_01_min/max_delay_03_min/config.yaml
11:49:43 [INFO] Saving config to /home/dominika/Desktop/deathOFb

1152 configs created


Load instance configuration

In [6]:
start_durations = {'07-00': ['16_h'], '18-00': ['2_h', '01_min', '30_min', '15_min', '05_min']}
delays = ['03_min', '05_min', '10_min']
# for start in starts:
#     for duration in durations:
#         for delay in delays:
#             generate_demand(city, start, duration, delay, df)
#             config_filepath = INSTANCE_PATH / f'{city}/instances/' / f'start_{start}' / f'duration_{duration}' / f'max_delay_{delay}' / 'config.yaml'
#             config_instance = load_instance_config(config_filepath)
#             instance_dir = os.path.dirname(config_filepath)
#             os.chdir(instance_dir)

Load filtered map nodes and edges

In [ ]:
duration = start_durations['07-00'][0]
config_filepath = INSTANCE_PATH / f'{city}/instances/' / f'start_07-00' / f'duration_{duration}' / f'max_delay_{delays[0]}' / 'config.yaml'
config_instance = load_instance_config(config_filepath)
instance_dir = os.path.dirname(config_filepath)
os.chdir(instance_dir)

20:31:58 [INFO] Loading instance config from /home/dominika/Desktop/deathOFbachelor/Instances/Sydney/instances/start_07-00/duration_16_h/max_delay_03_min/config.yaml


In [8]:
logging.info("Loading map")
map_nodes, map_edges = get_map(config_instance)

20:32:00 [INFO] Loading map
20:32:00 [INFO] Loading nodes from /home/dominika/Desktop/deathOFbachelor/Instances/Sydney/map/nodes.csv
/home/dominika/Desktop/deathOFbachelor/Ridesharing_DARP_instances/python/darpinstances/instance_generation/map.py:131: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  nodes = pd.read_csv(nodes_file_path, index_col=None, delim_whitespace=True)
20:32:00 [INFO] Loading edges from /home/dominika/Desktop/deathOFbachelor/Instances/Sydney/map/edges.csv
/home/dominika/Desktop/deathOFbachelor/Ridesharing_DARP_instances/python/darpinstances/instance_generation/map.py:139: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  edges = pd.read_csv(edges_file_path, index_col=None, delim_whitespace=True)


Load demand file

In [16]:
csvfile = RESOURCE_PATH / f'{city}_trips.csv'
df = load_data_from_csv(csvfile)

Set up nearest node provider

In [9]:
crs_metric = config_instance['map']['SRID']
nodes = map_nodes.to_crs(f'epsg:{crs_metric}')
nearest_node_provider = NearestNodeProvider(nodes)

In [10]:
# nodes
orig = wkt.loads(str(df['origin'][0]))
nearest_orig = nearest_node_provider.get_nearest_node(orig.x, orig.y)
nearest_orig
# nodes.loc[nearest_orig[0], 'db_id']

(625, 0.0008517854776950571)

Replace trip locations with nearest nodes

In [22]:
import pandas as pd

def replace_positions_with_nearest_nodes(demand_filepath, demand_data, nodesdata, nearest_node_provider, dataset):
    if os.path.exists(demand_filepath):
        demand_df = pd.read_csv(demand_filepath)
    else:
        demand_trips = []
        for _, row in demand_data.iterrows():
            orig = wkt.loads(str(row['origin']))
            nearest_orig = nearest_node_provider.get_nearest_node(orig.x, orig.y)
            orig_db_id = nodesdata.loc[nearest_orig[0], 'db_id']
            dest = wkt.loads(str(row['destination']))
            nearest_dest = nearest_node_provider.get_nearest_node(dest.x, dest.y)
            if nearest_orig[1] > 1000 or nearest_dest[1] > 1000:
                raise Exception("Further than we want")
            dest_db_id = nodesdata.loc[nearest_dest[0], 'db_id']
            origin_time = row['timestamp']
            trip = {
                "origin": orig_db_id,
                "destination": dest_db_id,
                "origin_time": origin_time,
                "dataset": dataset
            }
            demand_trips.append(trip)
        demand_df = pd.DataFrame(demand_trips)
        demand_df.to_csv(demand_filepath, index=False)
    return demand_df

## Load demand to DB

In [23]:
demand_filepath = RESOURCE_PATH / f'{city}_demand.csv'
dataset = 1
demand_df = replace_positions_with_nearest_nodes(demand_filepath, df, nodes, nearest_node_provider, dataset)

In [24]:
demand_df.shape

(287361, 4)

Load demand data into temp table in database

In [26]:
load_demand_to_db(demand_df, "tmp_demand", city.lower(), {'origin': BigInteger, 'destination': BigInteger, 'origin_time': TIMESTAMP, 'dataset': Integer})

20:54:04 [INFO] Starting _psycopg2 connection
20:54:04 [INFO] Starting sql_alchemy connection


Loading data into the database...
Data loaded into the database successfully.


Load data from tmp_demand to demand

In [27]:
insert_query = f"""INSERT INTO public.demand (origin, destination, origin_time, dataset)
    select origin, destination, origin_time, dataset from {city.lower()}.tmp_demand;"""
db.db.execute_sql(insert_query, schema='public')

- Set trip_location_set in db (and positions_set in config)
- Load trip locations into table

In [28]:
insert_query = f"""INSERT INTO public.trip_locations (request_id, origin, destination, set)
    select demand.id, origin, destination, trip_location_sets.id from public.demand, public.trip_location_sets
    where trip_location_sets.id = {config_instance['demand']['positions_set']} and dataset = {dataset};"""
db.db.execute_sql(insert_query, schema='public')

- Set trip_time_set in db (and time_set in config)
- Load trip times into table

In [29]:
insert_query = f"""INSERT INTO public.trip_times (request_id, time, set)
    select demand.id, origin_time, trip_time_sets.id from public.demand, public.trip_time_sets 
    where trip_time_sets.id = {config_instance['demand']['time_set']} and dataset = {dataset};"""
db.db.execute_sql(insert_query, schema='public')

## Generate distance matrix
Function `generate_dm()` calls executable `shortestPathsPreprocessor` which can be build with [ShoDi](https://github.com/aicenter/ShoDi).

- call didn't work in notebook, in terminal:

```shortestPathsPreprocessor -m dm -f xengraph --output-format hdf --preprocessing-mode slow --int-size 16 -i /home/dominika/Desktop/deathOFbachelor/Instances/Porto/map/map.xeng -o /home/dominika/Desktop/deathOFbachelor/Instances/Porto/dm```

In [12]:
generate_dm(config_instance, map_nodes, map_edges, False)

20:32:59 [INFO] Generating distance matrix in /home/dominika/Desktop/deathOFbachelor/Instances/Sydney/dm.h5.csv
20:32:59 [INFO] Using real speed from edges


/home/dominika/Desktop/deathOFbachelor/ShoDi/build/shortestPathsPreprocessor -m dm -f xengraph --output-format hdf --preprocessing-mode slow --int-size 16 -i /home/dominika/Desktop/deathOFbachelor/Instances/Sydney/map/map.xeng -o /home/dominika/Desktop/deathOFbachelor/Instances/Sydney/dm.h5


```console
>> shortestPathsPreprocessor -m dm -f xengraph --output-format hdf --preprocessing-mode slow --int-size 16 -i /home/dominika/Desktop/deathOFbachelor/Instances/Porto/map/map.xeng -o /home/dominika/Desktop/deathOFbachelor/Instances/Porto/dm
Computed 64746/64746 rows of the distance matrix.
Timer 'Distance Matrix preprocessing': Real time: 517.667
Storing the distance matrix.```

shortestPathsPreprocessor -m dm -f xengraph --output-format hdf --preprocessing-mode slow --int-size 16 -i /home/dominika/Desktop/deathOFbachelor/Instances/Sydney/map/map.xeng -o /home/dominika/Desktop/deathOFbachelor/Instances/Sydney/dm

In [4]:
import h5py
city = 'Porto'
dm_filepath = INSTANCE_PATH / f'{city}/dm.h5'
# dm_filepath = '/home/dominika/Desktop/deathOFbachelor/old_instances/Instances/DC/dm.h5'
with h5py.File(dm_filepath, 'r') as f:
    # dm = f['name-of-dataset'][:]
    dm = f['dm'][:]



In [42]:
os.chdir(PATH)

In [41]:
start_durations = {'07-00': ['16_h'], '18-00': ['2_h', '30_min', '15_min', '05_min', '01_min']}
delays = ['03_min', '05_min', '10_min']

duration = start_durations['07-00'][0]
# duration = start_durations['18-00'][0]
delay = delays[2]
config_filepath = INSTANCE_PATH / f'{city}/instances/' / f'start_07-00' / f'duration_{duration}' / f'max_delay_{delay}' / 'config.yaml'
config_instance = load_instance_config(config_filepath)
instance_dir = os.path.dirname(config_filepath)
os.chdir(instance_dir)

21:08:40 [INFO] Loading instance config from /home/dominika/Desktop/deathOFbachelor/Instances/Sydney/instances/start_07-00/duration_16_h/max_delay_10_min/config.yaml


In [48]:
for start in start_durations:
    for duration in start_durations[start]:
        for delay in delays:
            config_filepath = INSTANCE_PATH / f'{city}/instances/' / f'start_{start}' / f'duration_{duration}' / f'max_delay_{delay}' / 'config.yaml'
            if os.path.exists(INSTANCE_PATH / f'{city}/instances/' / f'start_{start}' / f'duration_{duration}' / f'max_delay_{delay}' / 'vehicles.csv'):
                continue
            logging.info(f"NEW INSTANCE for {config_filepath}")
            config_instance = load_instance_config(config_filepath)
            instance_dir = os.path.dirname(config_filepath)
            os.chdir(instance_dir)
            crs_metric = config_instance['map']['SRID_plane']
            nodes = map_nodes.to_crs(f'epsg:{crs_metric}')
            nearest_node_provider = NearestNodeProvider(nodes)
            generate_demand(map_nodes, config_instance, nearest_node_provider, crs_metric)
            desired_vehicle_count = 242
            generate_vehicles(map_nodes, config_instance, nearest_node_provider, desired_vehicle_count)
            os.chdir(PATH)


21:11:29 [INFO] NEW INSTANCE for /home/dominika/Desktop/deathOFbachelor/Instances/Sydney/instances/start_18-00/duration_01_min/max_delay_03_min/config.yaml
21:11:29 [INFO] Loading instance config from /home/dominika/Desktop/deathOFbachelor/Instances/Sydney/instances/start_18-00/duration_01_min/max_delay_03_min/config.yaml
21:11:29 [INFO] The demand file is already in /home/dominika/Desktop/deathOFbachelor/Instances/Sydney/instances/start_18-00/duration_01_min/max_delay_03_min/requests.csv, skipping demand generation.
21:11:29 [INFO] Saving vehicles to /home/dominika/Desktop/deathOFbachelor/Instances/Sydney/instances/start_18-00/duration_01_min/max_delay_03_min/vehicles.csv
/usr/lib/python3/dist-packages/pyproj/crs/crs.py:131: FutureWarning: '+init=<authority>:<code>' syntax is deprecated. '<authority>:<code>' is the preferred initialization method. When making the change, be mindful of axis order changes: https://pyproj4.github.io/pyproj/stable/gotchas.html#axis-order-changes-in-proj-6

## Generate trip requests

In [32]:
crs_metric = config_instance['map']['SRID_plane']
nodes = map_nodes.to_crs(f'epsg:{crs_metric}')
nearest_node_provider = NearestNodeProvider(nodes)

In [33]:
generate_demand(map_nodes, config_instance, nearest_node_provider, crs_metric)

20:57:12 [INFO] Generating demand
20:57:12 [INFO] Loading demand from DB
20:57:16 [INFO] 195626 requests fetched from db
20:57:16 [INFO] Assigning nearest nodes
20:57:16 [INFO] Saving requests to /home/dominika/Desktop/deathOFbachelor/Instances/Sydney/instances/start_07-00/duration_16_h/max_delay_10_min/requests.csv
/home/dominika/Desktop/deathOFbachelor/Ridesharing_DARP_instances/python/darpinstances/instance_generation/demand_generation.py:444: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  trips = pd.read_csv(trips_filepath, delim_whitespace=True)
20:57:16 [INFO] Saving instance file to ./trips.di
20:57:16 [INFO] Saving shapefile with pickups to: /home/dominika/Desktop/deathOFbachelor/Instances/Sydney/instances/start_07-00/duration_16_h/max_delay_10_min/shapefiles/pickup.shp
20:57:32 [INFO] Saving shapefile with drop offs to: /home/dominika/Desktop/deathOFbachelor/Instances/Sydney/instan

,origin,dest,time_ms
0,3619,60985,25200000
1,42718,24297,25200000
2,33099,38329,25200000
3,16937,29291,25200000
4,42718,40902,25200000
...,...,...,...
195621,62833,60056,82798000
195622,14611,40932,82799000
195623,19078,53359,82799000
195624,32142,7696,82799000


## Generate vehicles

In [34]:
logging.info("Generating vehicles")
# if 'vehicle_to_request_ratio' in config_instance['vehicles']:
#     desired_vehicle_count = int(len(requests) * config_instance['vehicles']['vehicle_to_request_ratio'])
# else:
#     desired_vehicle_count = config_instance['vehicles']['vehicle_count']
# desired_vehicle_count = 3998
# desired_vehicle_count = 400000
desired_vehicle_count = 217959
generate_vehicles(map_nodes, config_instance, nearest_node_provider, desired_vehicle_count)

20:58:36 [INFO] Generating vehicles
20:58:37 [INFO] Saving vehicles to /home/dominika/Desktop/deathOFbachelor/Instances/Sydney/instances/start_07-00/duration_16_h/max_delay_10_min/vehicles.csv
/usr/lib/python3/dist-packages/pyproj/crs/crs.py:131: FutureWarning: '+init=<authority>:<code>' syntax is deprecated. '<authority>:<code>' is the preferred initialization method. When making the change, be mindful of axis order changes: https://pyproj4.github.io/pyproj/stable/gotchas.html#axis-order-changes-in-proj-6
  in_crs_string = _prepare_from_proj_string(in_crs_string)
20:58:38 [INFO] Saving shapefile with vehicles to: /home/dominika/Desktop/deathOFbachelor/Instances/Sydney/instances/start_07-00/duration_16_h/max_delay_10_min/shapefiles/vehicles.shp


this also doesn't work:

In [5]:
# from darpbenchmark.sizing import calculate_sizing_for_instance
start_durations = {'07-00': ['16_h'], '18-00': ['2_h', '30_min', '15_min', '05_min', '01_min']}
delays = ['03_min', '05_min', '10_min']
methods = ['ih', 'vga']

start_time = '18-00'
duration = start_durations[start_time][0]
delay = delays[2]
method = methods[0]
results_path = RESULTS_PATH / f'{city}' / f'start_{start_time}' / f'duration_{duration}' / f'max_delay_{delay}' / method / 'config.yaml'
print('python3 python/scripts/sizing/instance_vehicle_count_sizing.py', results_path)
# calculate_sizing_for_instance(results_path)


python3 python/scripts/sizing/instance_vehicle_count_sizing.py /home/dominika/Desktop/deathOFbachelor/Results/Sydney/start_18-00/duration_2_h/max_delay_10_min/ih/config.yaml


run this in terminal instead:
```bash
python3 python/scripts/sizing/instance_vehicle_count_sizing.py /home/dominika/Desktop/deathOFbachelor/Results/Porto/start_07-00/duration_16_h/max_delay_03_min/ih/config.yaml

python3 python/scripts/sizing/instance_vehicle_count_sizing.py /home/dominika/Desktop/deathOFbachelor/Results/Sydney/start_07-00/duration_16_h/max_delay_03_min/ih/config.yaml
```